# Logistic Regression

### Predicting yes or no: who defaults, and who leaves?

Two business questions run through this notebook. A lender must decide which home equity applicants to approve, and where to set the cut-off when approving a customer who later defaults costs far more than turning away a good one. A telecoms retention team must decide which customers to ring with an offer, and how many calls are worth making. Both answers are yes or no, so a straight line will not do.

**How this notebook works**

- Each demo follows the same steps: load and look, run the model, read the output in plain English, check against JMP, say it to a manager.
- Python leads and JMP is the check. If the two disagree, question the Python (or the code you were given) first.
- GitHub Copilot is on. Use it, and check what it writes against your JMP result before you believe it.
- Run cells in order with **Shift+Enter**. The data files live in `../data/`.
- Every cost and retention figure in this notebook is invented for teaching. Real ones come from the business.

---
## Setup

In [ ]:
# The tools we need: pandas for tables, matplotlib for charts, statsmodels for the model,
# scikit-learn for the train/test split and the scoring measures
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from statsmodels.stats.outliers_influence import variance_inflation_factor

GREEN, AMBER = "#1A4C3B", "#E07B39"   # chart colours

# Show the data files that came with this repository
print(sorted(p.name for p in Path("../data").iterdir()))

### Tools used in both parts

Three small helpers do the repeated work, so the steps below stay readable. Read them once; you are not asked to change them.

- `odds_ratio_table` turns a fitted model into odds ratios with 95% intervals.
- `what_if` takes a base customer, changes one thing, and prints the chance of the outcome before and after. It is how you turn an odds ratio into a sentence a manager can use.
- `cutoff_table` shows what happens to the four cells of the confusion matrix as the cut-off moves.

In [ ]:
def odds_ratio_table(model):
    """Odds ratios with 95% intervals and p-values for a fitted statsmodels Logit."""
    ci = np.exp(model.conf_int())              # the model works in log-odds, so exponentiate to get odds ratios
    table = pd.DataFrame({
        "odds ratio": np.exp(model.params),
        "95% low": ci[0],
        "95% high": ci[1],
        "p-value": model.pvalues,
    })
    table["p-value"] = ["<0.0001" if p < 0.0001 else f"{p:.4f}" for p in table["p-value"]]
    return table.round({"odds ratio": 3, "95% low": 3, "95% high": 3})


def what_if(model, base, changes, base_label, new_label, outcome, digits=1):
    """Change one or more things about a base case and show the chance of the outcome before and after."""
    def chance(row):
        x = pd.DataFrame([row])[list(model.params.index[1:])]     # same columns, same order as the model
        return float(np.asarray(model.predict(sm.add_constant(x, has_constant="add")))[0])

    new = base.copy()
    for column, value in changes.items():
        new[column] = value
    p_base, p_new = chance(base), chance(new)

    # Odds = chance / (1 - chance). Quote the bigger over the smaller so the ratios read as "times".
    high, low = max(p_base, p_new), min(p_base, p_new)
    high_label, low_label = (new_label, base_label) if p_new >= p_base else (base_label, new_label)
    odds_ratio = (high / (1 - high)) / (low / (1 - low))
    chance_ratio = high / low

    print(f"{base_label}: {p_base:.1%} likely to {outcome}")
    print(f"{new_label}: {p_new:.1%}")
    print(f"Odds ratio ({high_label} against {low_label}): {odds_ratio:.{digits}f}")
    print(f"Ratio of the probabilities ({high_label} against {low_label}): {chance_ratio:.{digits}f}")
    print(f"For a manager: {high_label} has {odds_ratio:.{digits}f} times the odds, but only "
          f"{chance_ratio:.{digits}f} times the chance, of the outcome compared with {low_label}.")
    return p_base, p_new


def cutoff_table(y_true, p, cutoffs):
    """The confusion matrix and the usual measures at each cut-off. 'Positive' means the outcome we predict."""
    rows = []
    for c in cutoffs:
        tn, fp, fn, tp = confusion_matrix(y_true, (np.asarray(p) >= c).astype(int)).ravel()
        rows.append({"cut-off": c, "TN": tn, "FP": fp, "FN": fn, "TP": tp,
                     "accuracy": (tn + tp) / len(y_true), "precision": tp / (tp + fp),
                     "recall": tp / (tp + fn), "specificity": tn / (tn + fp)})
    return pd.DataFrame(rows).round(3)

---
## Demo: will this applicant default?

**The business question:** a lender wants to approve the good home equity applicants and decline the ones who will default. Which applicants should it approve, and where should it set the cut-off when a bad loan costs far more than a lost good customer?

The file holds 5,960 applicants. `BAD` is 1 if the applicant defaulted or became seriously delinquent, and 0 if they repaid. The file does not say which currency the amounts are in, so read them as currency units as recorded; the code shows them in thousands.

### Step 1: load and look

In [ ]:
# One row per applicant. BAD is the outcome we want to predict.
hmeq = pd.read_csv("../data/hmeq_full.csv")
print(hmeq.shape)
print(f"Share of applicants who defaulted (BAD = 1): {hmeq['BAD'].mean():.1%}")
hmeq.head()

In [ ]:
# Count the blanks in each column. A blank can carry information, as the
# default rates by missing value below show
hmeq.isna().sum()

### Step 2: a blank can tell you something

Before any model, look at who defaults. This is exploring only: no number from this step goes into the model.

In [ ]:
# Default rate where the debt-to-income figure is present and where it is blank
has_dti = hmeq["DEBTINC"].notna()
print(f"Debt-to-income present: {hmeq.loc[has_dti, 'BAD'].mean():.1%} defaulted")
print(f"Debt-to-income blank:   {hmeq.loc[~has_dti, 'BAD'].mean():.1%} defaulted")

In [ ]:
# Default rate by job (a blank job is kept as its own group) and by number of delinquent credit lines
by_job = hmeq.groupby(hmeq["JOB"].fillna("Missing"))["BAD"].mean().sort_values(ascending=False)
print((by_job * 100).round(1))
print()
lines = hmeq["DELINQ"].clip(upper=5)          # 5 stands for "5 or more"
print((hmeq.groupby(lines)["BAD"].mean() * 100).round(1))

A blank debt-to-income figure goes with a default rate of about 62%, against under 9% where the figure is present. The blank is a signal in its own right, so the model should be allowed to use it. Blanks do not always mean bad news: applicants with no job recorded default less than average.

### Step 3: prepare the data, split first

The split comes first. Everything the preparation learns from the data (here, the medians used to fill blanks) is worked out from the **training** rows only and then applied to the test rows. Work it out from all the rows and information about the test applicants leaks into the training, which flatters the model. That is called **leakage**.

The same five steps are used every time this file is modelled, so that models can be compared like for like.

In [ ]:
# 1. Split first: 70% to train the model, 30% held back to test it.
# stratify keeps the default rate the same in both parts; random_state fixes the split so everyone gets the same one.
X = hmeq.drop(columns="BAD")
y = hmeq["BAD"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)

print(f"Train: {len(X_train):,} applicants, {y_train.sum()} defaults ({y_train.mean():.2%})")
print(f"Test:  {len(X_test):,} applicants, {y_test.sum()} defaults ({y_test.mean():.2%})")

In [ ]:
# 2. Flag the blank debt-to-income figures BEFORE filling them, so the blank is not lost.
for part in (X_train, X_test):
    part["DEBTINC_missing"] = part["DEBTINC"].isna().astype(int)

# 3. Fill the other blanks in the number columns with the TRAINING median, for both parts.
numeric = ["LOAN", "MORTDUE", "VALUE", "YOJ", "DEROG", "DELINQ", "CLAGE", "NINQ", "CLNO", "DEBTINC"]
medians = X_train[numeric].median()           # worked out from the training rows only
X_train[numeric] = X_train[numeric].fillna(medians)
X_test[numeric] = X_test[numeric].fillna(medians)

In [ ]:
# 4. Blank REASON and JOB become their own category, "Missing"
# 5. Show the three money columns in thousands, so a coefficient is per 1,000 and not per unit
for part in (X_train, X_test):
    part["REASON"] = part["REASON"].fillna("Missing")
    part["JOB"] = part["JOB"].fillna("Missing")
    for column in ["LOAN", "MORTDUE", "VALUE"]:
        part[column] = part[column] / 1000

print(X_train[numeric].median().round(2))      # the training medians, money in thousands

In [ ]:
# Models need numbers, so turn REASON and JOB into 0/1 indicator columns.
# One level of each is left out as the reference: DebtCon for REASON, Other for JOB.
# Keep the readable versions too: they are exported for JMP below.
train_readable, test_readable = X_train.copy(), X_test.copy()

X_train = pd.get_dummies(X_train, columns=["REASON", "JOB"], dtype=int).drop(columns=["REASON_DebtCon", "JOB_Other"])
X_test = pd.get_dummies(X_test, columns=["REASON", "JOB"], dtype=int).drop(columns=["REASON_DebtCon", "JOB_Other"])
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)     # same columns, same order as the training table

print(X_train.shape, X_test.shape)
list(X_train.columns)

### Commentary

Why do the medians come from the training rows only, and what would go wrong if they came from all 5,960?

*Your commentary here*

### Step 4: why not a straight line?

Try the obvious thing first: fit an ordinary regression line to the 0/1 outcome and see what it predicts for the test applicants.

In [ ]:
# Ordinary least squares on a 0/1 outcome
line = sm.OLS(y_train, sm.add_constant(X_train)).fit()
line_pred = line.predict(sm.add_constant(X_test, has_constant="add"))

print(f"Predictions below 0: {(line_pred < 0).sum()}")
print(f"Predictions above 1: {(line_pred > 1).sum()}")
print(f"Range of predictions: {line_pred.min():.2f} to {line_pred.max():.2f}")

A chance of default cannot be below 0 or above 1, yet the line predicts both for hundreds of applicants. Logistic regression fixes this by modelling the **log-odds** instead. Probability, odds and log-odds are the same information on three scales:

| Scale | Example | Range |
|---|---|---|
| Probability | 20% chance of default | 0 to 1 |
| Odds | 1 to 4, written 0.25 | 0 to infinity |
| Log-odds | -1.39 | minus infinity to plus infinity |

A straight line works on the log-odds scale. Turning the answer back into a probability bends the line into an S, which can never leave 0 to 1.

### Step 5: fit the model

In [ ]:
# Add a constant (the intercept), then fit. The model predicts the chance that BAD = 1, a default.
X_train_c = sm.add_constant(X_train)
X_test_c = sm.add_constant(X_test, has_constant="add")

logit = sm.Logit(y_train, X_train_c).fit(disp=False)
print(f"Converged: {logit.mle_retvals['converged']}")
print(f"Pseudo R-squared: {logit.prsquared:.3f}   AIC: {logit.aic:,.1f}")

### Step 6: read the output in plain English

The model's coefficients are on the log-odds scale, which means little to a manager. Exponentiate them and you get **odds ratios**:

| Look for | What it says |
|---|---|
| Odds ratio of 1 | No effect on the odds |
| Above 1 | Each extra unit raises the odds of default (2 means the odds double) |
| Below 1 | Each extra unit lowers the odds (0.5 means the odds halve) |
| 95% interval | The range the true odds ratio plausibly lies in. If it contains 1, the effect could be zero |
| `DEBTINC_missing` | The odds ratio for a blank debt-to-income figure, compared with a figure being present |

In [ ]:
# Odds ratios with 95% intervals, per 1 unit of each driver (money columns per 1,000)
odds_ratio_table(logit)

An odds ratio of 2 **doubles the odds**. That is not the same as doubling the probability. The two are close only when the outcome is rare. A 5% chance has odds of about 1 to 19; double the odds and the chance becomes about 9.5%, close to double. Double the odds of an 80% chance and you reach 89%, nowhere near 160%.

### Step 7: the what-if translator

`what_if` shows what an odds ratio means in chances. Start with a **typical applicant**: every number column at its training median, the reference job and reason, and a debt-to-income figure present. Always name the base customer. The change in probability depends on where you start; the odds ratio does not.

In [ ]:
# The typical applicant: indicators off, number columns at the training median
typical = pd.Series(0.0, index=X_train.columns)
for column, value in X_train[numeric].median().items():
    typical[column] = value

row = sm.add_constant(pd.DataFrame([typical]), has_constant="add")[X_train_c.columns]
p_typical = float(np.asarray(logit.predict(row))[0])
print(f"A typical applicant has a {p_typical:.1%} chance of defaulting")

In [ ]:
# One more delinquent credit line (from 0 to 1)
_ = what_if(logit, typical, {"DELINQ": 1}, "Typical applicant", "Typical applicant with one more delinquent line", "default", digits=2)

In [ ]:
# Three delinquent lines, a debt-to-income of 50, and a blank debt-to-income figure
for changes, label in [({"DELINQ": 3}, "three delinquent lines"),
                       ({"DEBTINC": 50}, "debt-to-income of 50"),
                       ({"DEBTINC_missing": 1}, "debt-to-income blank")]:
    p0, p1 = what_if(logit, typical, changes, "Typical applicant", f"Typical applicant with {label}", "default", digits=2)
    print()

In [ ]:
# Protective factors work the same way. Flip the job to Office (odds ratio below 1):
# applicants in the reference job have 1 / 0.567 = 1.76 times the odds of an Office worker
_ = what_if(logit, typical, {"JOB_Office": 1}, "Typical applicant (reference job)", "Same applicant in an Office job", "default", digits=2)

### Step 8: the S-curve

The same model, drawn. The chance of default for the typical applicant as debt-to-income rises. It is an S, flat at both ends, steepest in the middle, and it never leaves 0 to 1.

In [ ]:
# Predicted chance of default as debt-to-income runs from 10 to 80, everything else typical
ratios = np.arange(10, 81, 1)
rows = pd.DataFrame([typical] * len(ratios))
rows["DEBTINC"] = ratios
curve = logit.predict(sm.add_constant(rows, has_constant="add")[X_train_c.columns])

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(ratios, curve, color=GREEN, linewidth=2.5)
ax.set_xlabel("Debt-to-income (points)")
ax.set_ylabel("Chance of default")
ax.set_ylim(0, 1)
plt.show()

### Step 9: test the model on applicants it has never seen

A model is judged on the **test** rows, which played no part in fitting it. The measure used first is **AUC**, the area under the ROC curve. It asks: pick one applicant who defaulted and one who repaid at random; how often does the model give the defaulter the higher chance? 1.0 is perfect, 0.5 is guessing.

In [ ]:
# The model's chance of default for every test and training applicant
p_test = logit.predict(X_test_c)
p_train = logit.predict(X_train_c)

auc_test = roc_auc_score(y_test, p_test)
auc_train = roc_auc_score(y_train, p_train)
print(f"AUC on the test applicants:     {auc_test:.3f}")
print(f"AUC on the training applicants: {auc_train:.3f}")

The two are almost the same. With about 20 drivers and over 4,000 training rows a logistic model hardly overfits. The gap between training and test matters more for flexible models, which is why it is worth printing every time.

How much does the blank-debt-to-income flag matter? Refit without it.

In [ ]:
# Same model, same split, but without the DEBTINC_missing flag
no_flag = [c for c in X_train_c.columns if c != "DEBTINC_missing"]
logit_no_flag = sm.Logit(y_train, X_train_c[no_flag]).fit(disp=False)
auc_no_flag = roc_auc_score(y_test, logit_no_flag.predict(X_test_c[no_flag]))
print(f"Test AUC without the flag: {auc_no_flag:.3f}   (with it: {auc_test:.3f})")

One flag moves the test AUC by about 0.09. Treating a blank as information was worth more than any tuning.

### Step 10: the confusion matrix and the accuracy trap

To turn a chance into a decision you need a **cut-off**: decline anyone whose chance of default is at or above it. At a cut-off of 0.5, count how each decision turns out.

In [ ]:
# Predict "default" when the chance is 0.5 or more. Rows are what happened, columns are what the model said.
decision = (p_test >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, decision).ravel()

matrix = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["Actually repaid", "Actually defaulted"],
    columns=["Approved (model says repay)", "Declined (model says default)"],
)
print(matrix)
print()
print(f"Good applicants turned away (false alarms):  {fp}")
print(f"Bad loans approved (missed defaults):         {fn}")

In [ ]:
# Accuracy is the share of decisions that were right. Compare it with a "model" that approves everyone.
accuracy = (tn + tp) / len(y_test)
always_repay = 1 - y_test.mean()
recall = tp / (tp + fn)

print(f"Accuracy of the model at 0.5:        {accuracy:.1%}")
print(f"Accuracy of approving everyone:      {always_repay:.1%}   (catches no defaults at all)")
print(f"Share of real defaults the model catches (recall): {recall:.1%}")

Approving everybody is right 80% of the time and catches nothing, so 88% accuracy is less impressive than it looks. When one outcome is rare, accuracy flatters the baseline. Two measures say what the model is actually doing:

- **Precision:** of the applicants the model declines, how many really default.
- **Recall** (also called sensitivity): of the applicants who really default, how many the model catches.

**Specificity** is the mirror image of recall: of the good applicants, how many are approved.

### Step 11: the cut-off is a dial

Moving the cut-off trades one kind of mistake for the other. Lower it and the model declines more people: it catches more defaults, and it turns away more good customers.

In [ ]:
# Five cut-offs, ending with 0.091, the cut-off that the invented costs imply
cutoff_table(y_test, p_test, [0.5, 0.3, 0.2, 0.15, 0.091])

### Step 12: put a price on being wrong

Which cut-off is best? Not the one with the highest accuracy. It is the one that costs the business least, and that depends on what each mistake costs.

**These costs are invented for teaching.** Suppose approving a loan that then defaults loses 10,000, and turning away a good applicant loses 1,000 of profit. Change the two numbers below and re-run to see how the answer moves.

In [ ]:
# INVENTED costs, for teaching only
cost_default = 10_000          # approve a loan that defaults
cost_turned_away = 1_000       # decline an applicant who would have repaid

def loan_cost(y_true, p, cutoff):
    """Total cost of the mistakes the cut-off makes."""
    y_true, p = np.asarray(y_true), np.asarray(p)
    missed_defaults = ((p < cutoff) & (y_true == 1)).sum()          # approved, then defaulted
    good_turned_away = ((p >= cutoff) & (y_true == 0)).sum()        # declined, would have repaid
    return missed_defaults * cost_default + good_turned_away * cost_turned_away

# The break-even cut-off: decline when the expected loss from approving beats the profit lost by declining
best_in_theory = cost_turned_away / (cost_default + cost_turned_away)
print(f"Cut-off from the costs: {cost_turned_away:,} / ({cost_default:,} + {cost_turned_away:,}) = {best_in_theory:.3f}")

for c in [0.5, 0.2, best_in_theory]:
    total = loan_cost(y_test, p_test, c)
    print(f"Cut-off {c:.3f}: cost {total:>12,.0f}   ({total / len(y_test):,.0f} per applicant)")

print(f"Approve everyone:  {y_test.sum() * cost_default:>12,.0f}")
print(f"Decline everyone:  {(y_test == 0).sum() * cost_turned_away:>12,.0f}")

In [ ]:
# Try every cut-off from 0.01 to 0.99 on the test applicants and plot the cost
grid = np.round(np.arange(0.01, 1.00, 0.01), 2)
costs = [loan_cost(y_test, p_test, c) for c in grid]
best = grid[int(np.argmin(costs))]
print(f"Cheapest cut-off on the grid: {best:.2f}, cost {min(costs):,.0f}")

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(grid, costs, color=GREEN, linewidth=2.5)
ax.scatter([0.5, best], [loan_cost(y_test, p_test, 0.5), min(costs)], color=AMBER, zorder=3)
ax.set_xlabel("Cut-off")
ax.set_ylabel("Total cost of mistakes")
plt.show()

In [ ]:
# Honest practice: choose the cut-off on the TRAINING applicants, then check it on the test applicants
train_costs = [loan_cost(y_train, p_train, c) for c in grid]
chosen = grid[int(np.argmin(train_costs))]
print(f"Cut-off chosen on the training applicants: {chosen:.2f}")
print(f"Its cost on the test applicants: {loan_cost(y_test, p_test, chosen):,.0f}")

The best cut-off is far below 0.5, and accuracy drops from 88% to about 75% to get there. That is the lesson: the business chooses the cut-off, and the cheapest decision is not the most accurate one. Notice too that the cost curve is flat near its lowest point, so a cut-off a little either side costs almost the same.

### Commentary

Accuracy fell from 88% to about 75% at the cheapest cut-off, yet the total cost fell by almost half. Explain how that can be, using the costs above.

*Your commentary here*

### Step 13: the ROC curve

The ROC curve draws the whole dial at once. Each point is one cut-off: across is the share of good applicants wrongly declined, up is the share of defaulters caught. A model that guesses lies on the diagonal; a better model bows up towards the top left. AUC is the area under the curve.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, p_test)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, color=GREEN, linewidth=2.5, label=f"Model (AUC {auc_test:.3f})")
ax.plot([0, 1], [0, 1], color="grey", linestyle="--", label="Guessing (AUC 0.5)")
for c in [0.5, 0.2, 0.091]:
    t_n, f_p, f_n, t_p = confusion_matrix(y_test, (p_test >= c).astype(int)).ravel()
    ax.scatter(f_p / (f_p + t_n), t_p / (t_p + f_n), color=AMBER, zorder=3)
    ax.annotate(f"cut-off {c}", (f_p / (f_p + t_n) + 0.02, t_p / (t_p + f_n) - 0.05))
ax.set_xlabel("Good applicants wrongly declined")
ax.set_ylabel("Defaulters caught")
ax.legend(loc="lower right")
plt.show()

---
## Exercise: check an assistant's code

An assistant was asked: *score the test applicants with the default model and report the AUC.* It wrote the code below. It runs without an error, and AUC is a perfectly normal thing to print.

Compare its AUC with the one you found in Step 9 and with the JMP result. Do they agree? If not, find the problem and fix it.

In [ ]:
# Assistant's answer: score the test applicants and report the AUC
def score_applicants(model, X):
    """Chance that each applicant defaults."""
    return 1 - model.predict(sm.add_constant(X, has_constant="add"))

p_assistant = score_applicants(logit, X_test)
print(f"Test AUC: {roc_auc_score(y_test, p_assistant):.3f}")

In [ ]:
# Your fix goes here

### Commentary

What went wrong, and what should an AUC well below 0.5 make you suspect?

*Your commentary here*

### Step 14: check against JMP

Export the prepared table with a `Split` column that marks each row as `Train` or `Test`, so JMP uses exactly the same split. Open `hmeq_prepared_with_split.csv` in JMP and follow the JMP workbook. The file also has a `Validation` column (0 for train, 1 for test) in case JMP asks for numbers.

In [ ]:
# One table for JMP: readable categories, the same filled values, the same split
export = pd.concat([
    train_readable.assign(BAD=y_train, Split="Train", Validation=0),
    test_readable.assign(BAD=y_test, Split="Test", Validation=1),
]).sort_index()
export = export[["BAD", "Split", "Validation"] + [c for c in export.columns if c not in ("BAD", "Split", "Validation")]]
export.to_csv("../data/hmeq_prepared_with_split.csv", index=False)
print(export.shape)
export.head()

Type the numbers from your JMP output into the cell below (leave a value as `None` if you did not record it), then run it. Odds ratios for the number columns should match. JMP codes categories differently, so compare its odds ratios for `JOB` as differences between pairs of jobs, not as the Python numbers directly. The fitted probabilities match either way.

In [ ]:
# Your JMP numbers go here
jmp = {"Test AUC": None, "DELINQ odds ratio": None, "DEBTINC odds ratio": None, "DEBTINC_missing odds ratio": None}

python = {"Test AUC": auc_test,
          "DELINQ odds ratio": np.exp(logit.params["DELINQ"]),
          "DEBTINC odds ratio": np.exp(logit.params["DEBTINC"]),
          "DEBTINC_missing odds ratio": np.exp(logit.params["DEBTINC_missing"])}

for name, value in python.items():
    other = jmp[name]
    verdict = "not recorded" if other is None else f"match: {np.isclose(other, value, atol=0.01)}"
    print(f"{name:28} Python {value:8.3f}   JMP {other}   {verdict}")

**If the signs in JMP are all the wrong way round,** JMP is modelling the other outcome. Set the Target Level to 1 (the default) before fitting.

### Say it to a manager

Two sentences, in business language, from your own output. Use a **what-if**: name the base applicant, change one thing, and give both chances of default. Do not quote a coefficient. Say how sure you are, or one limit on what the data can tell us.

The translator gives you the raw material:

In [ ]:
# Raw material for your two sentences: one what-if from the model
_ = what_if(logit, typical, {"DELINQ": 1}, "Typical applicant", "Typical applicant with one more delinquent line", "default", digits=2)

*Your two sentences here*

### Try this

Change one thing and re-run from the cell that needs it:

- Change `cost_default` to 5,000. Where does the cheapest cut-off move, and what happens to the gap in cost between 0.5 and the cheapest cut-off?
- Change the base applicant: set `typical["DEBTINC"] = 45` and re-run the what-ifs. Does the effect of one more delinquent line on the **chance** change? Does the odds ratio?
- Ask Copilot to build a table of the chance of default for 0, 1, 2, 3 and 4 delinquent lines, and check the 0, 1 and 3 values against the what-ifs above.

---
## Exercise: who should the retention team ring?

**The business question:** a telecoms retention team can ring customers with an offer, but each call costs money. Which customers should it ring, and how many calls are worth making?

The file holds 7,043 customers. `Churn` is `Yes` for the 26.5% who left. The file has no `OnlineSecurity` or `TechSupport` columns, so the model cannot use them.

### Step 1: load and fix

In [ ]:
# The data sheet has a name; the other sheets in the workbook are summaries
telco = pd.read_excel("../data/Telco-Customer-Churn.xlsx", sheet_name="Telco-Customer-Churn")
print(telco.shape)
telco.dtypes

In [ ]:
# TotalCharges arrives as text because 11 cells are blank. errors="coerce" turns the blanks into missing values.
telco["TotalCharges"] = pd.to_numeric(telco["TotalCharges"], errors="coerce")
print("Blank TotalCharges:", telco["TotalCharges"].isna().sum())
print("Their tenure in months:", telco.loc[telco["TotalCharges"].isna(), "tenure"].unique())

# The outcome as 0/1
churn = (telco["Churn"] == "Yes").astype(int)
print(f"Churn rate: {churn.mean():.1%}")

The 11 blanks are new customers (tenure 0) who have not been billed yet, so blank means "nothing billed", not "unknown".

**Do not use `TotalCharges` as a driver.** It is roughly tenure multiplied by the monthly charge, so it moves almost in step with tenure and would hide the tenure effect:

In [ ]:
print(telco[["tenure", "MonthlyCharges", "TotalCharges"]].corr().round(3))

### Step 2: look at who leaves

In [ ]:
# Churn rate by contract, internet service, payment method and tenure band
for column in ["Contract", "InternetService", "PaymentMethod"]:
    print((churn.groupby(telco[column]).mean() * 100).round(1))
    print()

bands = pd.cut(telco["tenure"], [-1, 6, 12, 24, 48, 72], labels=["0-6", "7-12", "13-24", "25-48", "49-72"])
print((churn.groupby(bands, observed=True).mean() * 100).round(1))

Month-to-month customers leave at 42.7%, two-year customers at 2.8%. Fibre customers leave far more than DSL, and so do customers who pay by electronic cheque. New customers are the most likely to go.

### Step 3: the dummy variable trap

Some columns say the same thing twice. Every customer with `InternetService = "No"` also has `StreamingMovies = "No internet service"`:

In [ ]:
print(pd.crosstab(telco["InternetService"], telco["StreamingMovies"]))

If both went into the model, one of them would be a copy of the other, and the model cannot tell them apart. The same happens if you keep an indicator column for **every** level of a category as well as the intercept: the columns add up to the constant. That is the **dummy variable trap**. Leave one level out as the reference.

In [ ]:
# The trap in numbers: an indicator for every level of InternetService, plus the intercept
levels = pd.get_dummies(telco["InternetService"], dtype=int)
levels["const"] = 1
print("Columns:", levels.shape[1], "  Independent columns (rank):", np.linalg.matrix_rank(levels.values))

trap = sm.Logit(churn, levels).fit(disp=False)
print(trap.bse.round(0))      # standard errors in the hundreds of thousands: the columns cannot be told apart

Four columns, but only three independent ones, so the standard errors are enormous and the odds ratios mean nothing. The exercise drops `StreamingMovies` (a copy of the internet service information), `MultipleLines` (its "No phone service" level repeats the phone service column) and `gender` (not a fair basis for treating customers differently).

### Step 4: build the columns and split

The columns below are built by fixed rules (yes or no, which contract), and nothing is learned from the data, so building them before the split does not leak. The reference levels are no, month-to-month, DSL and bank transfer. `MonthlyCharges` is per $10 so that its odds ratio is for a $10 change. (The charges in this file are in US dollars.)

In [ ]:
# Fixed rules only: yes/no columns, and one indicator per non-reference level
X_t = pd.DataFrame({
    "tenure": telco["tenure"],
    "MonthlyCharges_per10": telco["MonthlyCharges"] / 10,
    "SeniorCitizen": telco["SeniorCitizen"],
    "Partner": (telco["Partner"] == "Yes").astype(int),
    "Dependents": (telco["Dependents"] == "Yes").astype(int),
    "PaperlessBilling": (telco["PaperlessBilling"] == "Yes").astype(int),
    "Contract_OneYear": (telco["Contract"] == "One year").astype(int),
    "Contract_TwoYear": (telco["Contract"] == "Two year").astype(int),
    "Internet_Fibre": (telco["InternetService"] == "Fiber optic").astype(int),
    "Internet_None": (telco["InternetService"] == "No").astype(int),
    "Pay_CreditCard": (telco["PaymentMethod"] == "Credit card (automatic)").astype(int),
    "Pay_ElectronicCheque": (telco["PaymentMethod"] == "Electronic check").astype(int),
    "Pay_MailedCheque": (telco["PaymentMethod"] == "Mailed check").astype(int),
})

# The same split call as before
X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_t, churn, test_size=0.3, random_state=1, stratify=churn)
print(f"Train: {len(X_train_t):,} customers, {y_train_t.sum()} churners ({y_train_t.mean():.2%})")
print(f"Test:  {len(X_test_t):,} customers, {y_test_t.sum()} churners ({y_test_t.mean():.2%})")

### Step 5: fit and read the odds ratios

In [ ]:
X_train_tc = sm.add_constant(X_train_t)
X_test_tc = sm.add_constant(X_test_t, has_constant="add")

logit_t = sm.Logit(y_train_t, X_train_tc).fit(disp=False)
print(f"Converged: {logit_t.mle_retvals['converged']}")
print(f"Pseudo R-squared: {logit_t.prsquared:.3f}   AIC: {logit_t.aic:,.1f}")
odds_ratio_table(logit_t)

Read it as before. Tenure: each extra month multiplies the odds of leaving by about 0.97, so a customer who has stayed a year has far lower odds than a new one. A two-year contract multiplies the odds by about 0.21 compared with month-to-month. Fibre multiplies them by about 2.4 compared with DSL. Partner, mailed cheque and credit card have intervals that contain 1, so the data cannot separate them from no effect.

### Step 6: when a driver mostly stands in for others

`MonthlyCharges` looks like a strong driver on its own. Watch what happens when the other drivers join it.

In [ ]:
# MonthlyCharges on its own, then with everything else in the model
alone = sm.Logit(y_train_t, sm.add_constant(X_train_t[["MonthlyCharges_per10"]])).fit(disp=False)
print(f"On its own:   odds ratio {np.exp(alone.params['MonthlyCharges_per10']):.2f} per $10, p = {alone.pvalues['MonthlyCharges_per10']:.1e}")
print(f"With the rest: odds ratio {np.exp(logit_t.params['MonthlyCharges_per10']):.3f} per $10, p = {logit_t.pvalues['MonthlyCharges_per10']:.2f}")

# The variance inflation factor measures how much a driver overlaps with the others
vif = pd.Series([variance_inflation_factor(X_train_tc.values, i) for i in range(1, X_train_tc.shape[1])],
                index=X_train_t.columns).round(1)
print()
print(vif.sort_values(ascending=False).head(4))

On its own, each extra $10 raises the odds of leaving by about 17%. With contract, internet service and tenure in the model the effect nearly vanishes and could be zero. The monthly charge is mostly a proxy for what the customer bought, and its variance inflation factor is the highest in the model.

### Step 7: the what-if translator, again

Base customer: 12 months of tenure, $70 a month, DSL, month-to-month, nothing else switched on, paying by bank transfer.

In [ ]:
# The base customer: everything off except tenure and the monthly charge
base_t = pd.Series(0.0, index=X_train_t.columns)
base_t["tenure"] = 12
base_t["MonthlyCharges_per10"] = 7.0

# Month-to-month against a two-year contract
p_m2m, p_two = what_if(logit_t, base_t, {"Contract_TwoYear": 1},
                       "Month-to-month customer", "Two-year contract customer", "leave", digits=1)

In [ ]:
# One-year contract, then fibre with electronic cheque
what_if(logit_t, base_t, {"Contract_OneYear": 1}, "Month-to-month customer", "One-year contract customer", "leave", digits=1)
print()
_ = what_if(logit_t, base_t, {"Internet_Fibre": 1, "Pay_ElectronicCheque": 1},
        "Base customer (DSL, bank transfer)", "Same customer on fibre, paying by electronic cheque", "leave", digits=1)

The same question can give four different numbers, depending on what you mean. Say which one you are quoting.

In [ ]:
# Month-to-month against two-year, four ways
rate = churn.groupby(telco["Contract"]).mean()
raw_rate_ratio = rate["Month-to-month"] / rate["Two year"]
raw_odds_ratio = (rate["Month-to-month"] / (1 - rate["Month-to-month"])) / (rate["Two year"] / (1 - rate["Two year"]))
adjusted_odds_ratio = 1 / np.exp(logit_t.params["Contract_TwoYear"])
adjusted_chance_ratio = p_m2m / p_two

print(f"Raw data, ratio of churn rates (42.7% against 2.8%): {raw_rate_ratio:5.1f}")
print(f"Raw data, odds ratio:                                {raw_odds_ratio:5.1f}")
print(f"Model, odds ratio (other drivers held constant):     {adjusted_odds_ratio:5.1f}")
print(f"Model, ratio of chances for the base customer:       {adjusted_chance_ratio:5.1f}")

### Commentary

Why does the model's odds ratio (about 4.7) come out so much smaller than the raw one (about 26)?

*Your commentary here*

### Step 8: test the model, and the baseline to beat

In [ ]:
p_test_t = logit_t.predict(X_test_tc)
p_train_t = logit_t.predict(X_train_tc)

print(f"Test AUC:  {roc_auc_score(y_test_t, p_test_t):.3f}")
print(f"Train AUC: {roc_auc_score(y_train_t, p_train_t):.3f}")
print(f"Accuracy of predicting that nobody leaves: {1 - y_test_t.mean():.1%}")

# What a single driver manages on its own
for cols in (["tenure"], ["Contract_OneYear", "Contract_TwoYear"]):
    single = sm.Logit(y_train_t, sm.add_constant(X_train_t[cols])).fit(disp=False)
    auc = roc_auc_score(y_test_t, single.predict(sm.add_constant(X_test_t[cols], has_constant="add")))
    print(f"Test AUC with {cols} alone: {auc:.3f}")
print(f"If the AUC had read {1 - roc_auc_score(y_test_t, p_test_t):.3f}, the model would be upside down.")

In [ ]:
# The confusion matrix measures at five cut-offs. Positive means "leaves".
cutoff_table(y_test_t, p_test_t, [0.5, 0.4, 0.3, 0.25, 0.2])

### Step 9: the economics set the cut-off

**These figures are invented for teaching.** An offer costs 30 per customer contacted. A customer saved from leaving is worth 300. The offer saves 40% of the customers who would otherwise have left. So contact a customer if their chance of leaving times 0.4 times 300 beats 30, which means a chance above 0.25.

In [ ]:
# INVENTED retention economics, for teaching only
offer_cost = 30
saved_value = 300
save_rate = 0.4

def retention_profit(y_true, p, cutoff):
    """Profit from contacting every customer whose chance of leaving is at or above the cut-off."""
    y_true, p = np.asarray(y_true), np.asarray(p)
    contacted = p >= cutoff
    saved = y_true[contacted].sum() * save_rate           # real leavers among those contacted, times the save rate
    return saved * saved_value - contacted.sum() * offer_cost

print(f"Break-even cut-off: {offer_cost} / ({save_rate} x {saved_value}) = {offer_cost / (save_rate * saved_value):.2f}")
for c in [0.5, 0.25]:
    print(f"Cut-off {c}: profit {retention_profit(y_test_t, p_test_t, c):>8,.0f}, customers contacted {(p_test_t >= c).sum():,}")
print(f"Contact everyone: profit {retention_profit(y_test_t, p_test_t, 0.0):,.0f}")

### Step 10: ask Copilot for the profit curve

Write a short prompt that tells Copilot four things: the function `retention_profit`, the true outcomes `y_test_t`, the model's chances `p_test_t`, and what you want (a line chart of test profit for every cut-off from 0.05 to 0.95 in steps of 0.01, with the best cut-off marked and printed). Paste the code it gives you in the cell below and run it.

Then check it. The best cut-off should be close to 0.29, with a profit of about 27,600. If yours is not, find out why before you trust it.

In [ ]:
# Your Copilot code goes here

### Commentary

The profit curve is fairly flat near its highest point. What does that tell a retention manager about how precisely the cut-off needs to be chosen, and about the figures it rests on?

*Your commentary here*

### Step 11: check against JMP

Export the prepared table, with the same `Split` column, and follow the JMP workbook. Set the Target Level to `Yes` before fitting.

In [ ]:
# One table for JMP: readable categories, the same split
telco_export = pd.DataFrame({
    "customerID": telco["customerID"],
    "Churn": telco["Churn"],
    "tenure": telco["tenure"],
    "MonthlyCharges_per10": telco["MonthlyCharges"] / 10,
    "SeniorCitizen": np.where(telco["SeniorCitizen"] == 1, "Yes", "No"),
    "Partner": telco["Partner"],
    "Dependents": telco["Dependents"],
    "PaperlessBilling": telco["PaperlessBilling"],
    "Contract": telco["Contract"],
    "InternetService": telco["InternetService"],
    "PaymentMethod": telco["PaymentMethod"],
})
telco_export["Split"] = np.where(telco_export.index.isin(X_train_t.index), "Train", "Test")
telco_export["Validation"] = (telco_export["Split"] == "Test").astype(int)
telco_export = telco_export[["Churn", "Split", "Validation"] + [c for c in telco_export.columns if c not in ("Churn", "Split", "Validation")]]
telco_export.to_csv("../data/telco_prepared_with_split.csv", index=False)
print(telco_export["Split"].value_counts())
telco_export.head()

In [ ]:
# Your JMP numbers go here
jmp_t = {"Test AUC": None, "Two-year odds ratio": None, "Fibre odds ratio": None, "Tenure odds ratio": None}

python_t = {"Test AUC": roc_auc_score(y_test_t, p_test_t),
            "Two-year odds ratio": np.exp(logit_t.params["Contract_TwoYear"]),
            "Fibre odds ratio": np.exp(logit_t.params["Internet_Fibre"]),
            "Tenure odds ratio": np.exp(logit_t.params["tenure"])}

for name, value in python_t.items():
    other = jmp_t[name]
    verdict = "not recorded" if other is None else f"match: {np.isclose(other, value, atol=0.01)}"
    print(f"{name:22} Python {value:7.3f}   JMP {other}   {verdict}")

### Say it to a manager

Two sentences, in business language, from your own output. Use a **what-if**: name the base customer, change one thing, and give both chances of leaving. Do not quote a coefficient. Say how the offer should be targeted, or one limit on what the data can tell us.

The translator gives you the raw material:

In [ ]:
# Raw material for your two sentences: one what-if from the model
_ = what_if(logit_t, base_t, {"Contract_TwoYear": 1}, "Month-to-month customer", "Two-year contract customer", "leave", digits=1)

*Your two sentences here*

---
## Stretch

A blank space for quick work. Use Copilot to take the analysis further, then check the numbers against JMP. Some ideas: refit the Telco model without `MonthlyCharges` and compare the fibre odds ratio and the test AUC; plot the chance of leaving against tenure for a month-to-month and a two-year customer; find the ten customers the model is most sure will leave and ask what they have in common; or refit the loan model with `DEROG` and `DELINQ` as groups instead of counts.